## Testing edits to reassortment MLE code

In [4]:
import numpy as np
from scipy.linalg import expm
from scipy.optimize import minimize_scalar
import pandas as pd
import ast
import tskit

In [2]:
class SCAR(object):
    
    """
        Model class for the structured coalescent with ancestral recombination (SCAR)
        Ancestral states can be given (known) or marginalized over (unknown).
        This version relaxes rules about the ARG:
        Specifically coal nodes need not be strictly bifurcating and can have any number of children.
        But all children must have one and only one parent.
    """
    
    def __init__(self,reassortment_rate,M,Ne,genome_length,**kwargs):
        
        '''             
            Parameters: 
                reassortment_rate (float): reassortment rate per lineage per site
                M (2D list/array): migration rates (forward-time) between subpopulations 
                Ne (1D list/array): effective population size of each subpopulation
                genome_length (int): The number of segments K, with unit segments [0,1), ..., [K-1, K).
               
            Optional keyword arguments:
                bounds (tuple): lower and upper bounds on estimated parameter given as (lower,upper)
                dt_step (str): integration time step used to compute lineage state probabilities
                known_ancestral_states (boolean): ancestral states need to be given in tree series (ts) if True
        '''
        
        # Model params
        self.reassortment_rate = reassortment_rate
        self.M = np.transpose(np.array(M)) # transpose to get reverse time matrix        
        self.Ne = np.array(Ne)
        self.genome_length = genome_length
        
        # Likelihood calculation params
        self.bounds = kwargs.get('bounds', (0, np.inf))
        self.dt_step = kwargs.get('dt_step', 0.1)
        self.known_ancestral_states = kwargs.get('known_ancestral_states', False)
        

In [6]:
# Read in the tskit object!

samples = 10
genome_length = 3
reassortment_rate = 0.01
Ne = 10  # effective pop sizes
#M = [[0.0,0.25],[0.25,0.0]]  # migration rate matrix
M = [[0]]

ts = tskit.load("arg_w_tables.trees")

# Initialize SCAR model class
bounds = (0.0,0.1)
scar_model = SCAR(reassortment_rate,M,Ne,genome_length,bounds=bounds)

# Check numerical optimization for MLE of single param
#mle = scar_model.opt_MLE(ts)
#print(mle)

In [9]:
#    def compute_neg_log_like(self,reassortment_rate,table_nodes,table_edges):
self = scar_model
reassortment_rate = 0.01
table_nodes = "arg_w_tables.trees.nodes.csv"
table_edges = "arg_w_tables.trees.edges.csv"

In [10]:
pops = 1
if pops == 0: # we never assigned populations
    pops = 1
#samples = ts.num_samples
#states = [st.id for st in ts.populations()]

# Get transition rate matrix
Q = self.M - np.diag(np.sum(self.M,axis=1)) # set diagonals to negative row sums

# Read in tables of nodes and edges for ARG
nodes_df = pd.read_csv(table_nodes)
edges_df = pd.read_csv(table_edges)
    

In [14]:
children = edges_df["child"].to_numpy()
parents = edges_df["parent"].to_numpy()
segments = edges_df["segments"].apply(ast.literal_eval).to_numpy()


In [15]:
children

array([ 3, 10,  1,  5,  4,  8, 12, 10,  7,  6, 11, 13,  9,  2,  0, 17, 17,
       18, 14, 19, 16, 20, 15, 21, 22, 23, 21])

In [16]:
parents

array([10, 11, 11, 12, 12, 13, 13, 14, 14, 15, 15, 16, 16, 17, 18, 18, 19,
       19, 20, 20, 21, 22, 22, 23, 23, 24, 24])

In [17]:
segments

array([list([0, 1, 2]), list([1, 2]), list([0, 1, 2]), list([0, 1, 2]),
       list([0, 1, 2]), list([0, 1, 2]), list([0, 1, 2]), list([0]),
       list([0, 1, 2]), list([0, 1, 2]), list([0, 1, 2]), list([0, 1, 2]),
       list([0, 1, 2]), list([0, 1, 2]), list([0, 1, 2]), list([1, 2]),
       list([0]), list([0, 1, 2]), list([0, 1, 2]), list([0, 1, 2]),
       list([0, 1, 2]), list([0, 1, 2]), list([0, 1, 2]), list([0, 1]),
       list([0, 1, 2]), list([2]), list([2])], dtype=object)

In [41]:
for i in np.arange(0,len(nodes_df)):
    event_time = nodes_df.time[i]
    if (i+1 < len(nodes_df)): # if not at final event
        next_time = nodes_df.time[i+1]
    else:
        next_time = nodes_df.time[i]
    t_elapsed = next_time - event_time # time elapsed between events

In [53]:
i=10
event_time = nodes_df.time[i]
if (i+1 < len(nodes_df)): # if not at final event
    next_time = nodes_df.time[i+1]
else:
    next_time = nodes_df.time[i]
t_elapsed = next_time - event_time # time elapsed between events

In [54]:
event_type = None
if nodes_df.type[i] == 'sample':
    event_type = 'sample'
if nodes_df.type[i] == 'coalescence':
    event_type = 'coalescent'
if nodes_df.type[i] == 'recombination':
    event_type = 'recombination'

In [55]:
event_type

'recombination'

In [60]:
edges_df

,parent,child,segments
0,10,3,"[0, 1, 2]"
1,11,10,"[1, 2]"
2,11,1,"[0, 1, 2]"
3,12,5,"[0, 1, 2]"
4,12,4,"[0, 1, 2]"
5,13,8,"[0, 1, 2]"
6,13,12,"[0, 1, 2]"
7,14,10,[0]
8,14,7,"[0, 1, 2]"
9,15,6,"[0, 1, 2]"


In [52]:
nodes_df

,id,time,type,meta
0,0,0.000000,sample,NaN
1,1,0.000000,sample,NaN
2,2,0.000000,sample,NaN
3,3,0.000000,sample,NaN
4,4,0.000000,sample,NaN
5,5,0.000000,sample,NaN
6,6,0.000000,sample,NaN
7,7,0.000000,sample,NaN
8,8,0.000000,sample,NaN
9,9,0.000000,sample,NaN


In [56]:
event_prob = 1.0
prob_no_coal = 1.0
prob_no_mig = 1.0
prob_no_reassort = 1.0

In [58]:
#if 'recombination' == event_type:
# Find child of parent node
child = children[parents == i]
child = np.unique(child)
assert len(child) == 1

In [59]:
child

array([3])

In [64]:
child = np.array(10)
recomb_parents = parents[children == child]


In [65]:
recomb_parents

array([11, 14])